# LIMPEZA DE DADOS E DIVIR OS DADOS DE TREINO/TESTE

Esse notebook basicamente faz a leitura do `RESULTADOS_2025.csv`, aplica os filtros, sorteia 8 000 linhas e separa 80% dos dados para treino e 20% para o teste. A variâvel alvo da regressão é `NU_NOTA_MT`, que é a nota de matemática do estudante no enem.

# 1. Parâmetros
Definição dos parâmetros

In [6]:
import csv
from pathlib import Path

import numpy as np
from machinelearning_mnpkllps.preprocessing import train_test_split

CSV = Path("../data/RESULTADOS_2025.csv")
PROC = Path("../data")     # onde serão salvos treino.csv e teste.csv
N_SAMPLE = 8000            # linhas da amostra Total de linhas: 4810772
TEST_SIZE = 0.2            # fração do teste
SEED = 20251               # semente do sorteio da amostra e do split

TARGET = "NU_NOTA_MT"
NUMERIC = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_COMP1", "NU_NOTA_COMP2", "NU_NOTA_COMP3","NU_NOTA_COMP4","NU_NOTA_COMP5"]
CATEGORICAL = ["SG_UF_ESC", "TP_LINGUA"]
FEATURES = NUMERIC + CATEGORICAL

# 2. APLICANDO OS FILTROS E PREPARANDO OS DADOS

Só as colunas acima (e as de presença/status da redação, usadas nos filtros) são lidas. 


Filtros: (1) presente nas 4 provas, (2) redação sem problemas, (3) escola informada (dependência, localização e UF) e língua (espanhol/inglês) informada. A amostra é **uniforme**: fazendo reservoir sampling com semente, que sorteia `N_SAMPLE` linhas.

In [7]:
def num(s):
    return float(s.replace(",", "."))      # aceita decimal com vírgula

rng = np.random.default_rng(SEED)
amostra, vistas = [], 0
funil = {"Total de linhas": 0, "Presente nas 4 provas": 0, "Redação sem problemas": 0, "Lingua informada": 0}

with open(CSV, encoding="latin-1", newline="") as f: #esse csv ta com encoding diferente
    leitor = csv.reader(f, delimiter=";")
    col = {nome.strip(): i for i, nome in enumerate(next(leitor))}
    i_pres = [col[c] for c in ("TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT")]
    i_status = col["TP_STATUS_REDACAO"]
    i_num = [col[c] for c in NUMERIC + [TARGET]]
    i_cat = [col[c] for c in CATEGORICAL]

    for linha in leitor:
        if len(linha) < len(col):                         # ignora linha truncada
            continue
        funil["Total de linhas"] += 1
        if any(linha[j] != "1" for j in i_pres):          # filtro 1  ESTUDANTE QUE SE AUSENTOU EM UMA PROVA
            continue
        funil["Presente nas 4 provas"] += 1
        if linha[i_status] != "1":                        # filtro 2   A REDAÇÃO TEVE ALGUM PROBLEMA
            continue
        funil["Redação sem problemas"] += 1
        uf, lingua = (linha[j].strip() for j in i_cat)
        if "" in (uf, lingua):                          # filtro 3 LÍNGUA INFORMADA
            continue
        funil["Lingua informada"] += 1

        cn, ch, lc, red1, red2, red3, red4, red5, mt = (num(linha[j]) for j in i_num) # PADRONIZANDO DOS DADOS NUMÉRICOS
        registro = [cn, ch, lc, red1, red2, red3, red4, red5, uf, int(num(lingua)), mt]

        # Usa o reservoir sampling onde as N_SAMPLE primeiras entram e depois cada uma substitui uma da amostra
        if vistas < N_SAMPLE:
            amostra.append(registro)
        else:
            j = int(rng.integers(0, vistas + 1))
            if j < N_SAMPLE:
                amostra[j] = registro
        vistas += 1

total = funil["Total de linhas"]
sem_problemas = funil["Lingua informada"]

print(f"\nEstudantes sem problemas em nenhum dos critérios: "
      f"{100 * sem_problemas / total:.2f}%")


Estudantes sem problemas em nenhum dos critérios: 25.05%


## 3. Split treino/teste e arquivos

Split com a função da biblioteca e a mesma semente. A análise exploratória deve olhar apenas `treino.csv`.

In [8]:
dados = np.array(amostra, dtype=object)           # colunas: FEATURES + [TARGET]
X, y = dados[:, :-1], dados[:, -1].astype(float) #DIVIDINDO, A ÚLTIMA COLUNA É O ALVO
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, seed=SEED)

def salvar(X, y, nome):
    with open(PROC / nome, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(FEATURES + [TARGET]) #concatenação de listas
        w.writerows(np.column_stack([X, y]).tolist()) #concatena as colunas e dpoies trandforma em lista

salvar(Xtr, ytr, "treino.csv")
salvar(Xte, yte, "teste.csv")
print(f"treino = {len(ytr):,}  teste = {len(yte):,}  salvos em {PROC}".replace(",", "."))

treino = 6.400  teste = 1.600  salvos em ..\data
